In [1]:
import os
from pathlib import Path
import pandas as pd
import re
from sklearn.model_selection import train_test_split

from torch.utils.data import Dataset
from PIL import Image
import torch

# Loading Data

In [2]:
DATA_ROOT = Path("Data")
BENIGN_DIR = DATA_ROOT / "benign"
MALIGNANT_DIR = DATA_ROOT / "malignant"

In [3]:
records = []
print(len(records))

0


In [4]:
# Load Benign Images
for img_path in BENIGN_DIR.glob("*.png"):
    print(img_path)
    records.append({
        "path": str(img_path),
        "filename": img_path.name,
        "label": 0,
        "label_name": "benign"
    })

print(len(records))

Data\benign\SOB_B_A-14-22549AB-100-001.png
Data\benign\SOB_B_A-14-22549AB-100-002.png
Data\benign\SOB_B_A-14-22549AB-100-003.png
Data\benign\SOB_B_A-14-22549AB-100-004.png
Data\benign\SOB_B_A-14-22549AB-100-005.png
Data\benign\SOB_B_A-14-22549AB-100-006.png
Data\benign\SOB_B_A-14-22549AB-100-007.png
Data\benign\SOB_B_A-14-22549AB-100-008.png
Data\benign\SOB_B_A-14-22549AB-100-009.png
Data\benign\SOB_B_A-14-22549AB-100-010.png
Data\benign\SOB_B_A-14-22549AB-100-011.png
Data\benign\SOB_B_A-14-22549AB-100-012.png
Data\benign\SOB_B_A-14-22549AB-100-013.png
Data\benign\SOB_B_A-14-22549AB-100-014.png
Data\benign\SOB_B_A-14-22549AB-100-015.png
Data\benign\SOB_B_A-14-22549AB-100-016.png
Data\benign\SOB_B_A-14-22549AB-100-017.png
Data\benign\SOB_B_A-14-22549AB-100-018.png
Data\benign\SOB_B_A-14-22549AB-100-019.png
Data\benign\SOB_B_A-14-22549AB-100-020.png
Data\benign\SOB_B_A-14-22549AB-100-021.png
Data\benign\SOB_B_A-14-22549AB-100-022.png
Data\benign\SOB_B_A-14-22549AB-100-023.png
Data\benign

In [5]:
# Load Malignant Images
for img_path in MALIGNANT_DIR.glob("*.png"):
    print(img_path)
    records.append({
        "path": str(img_path),
        "filename": img_path.name,
        "label": 0,
        "label_name": "malignant"
    })

    print(len(records))

Data\malignant\SOB_M_DC-14-10926-100-001.png
2480
Data\malignant\SOB_M_DC-14-10926-100-002.png
2481
Data\malignant\SOB_M_DC-14-10926-100-003.png
2482
Data\malignant\SOB_M_DC-14-10926-100-004.png
2483
Data\malignant\SOB_M_DC-14-10926-100-005.png
2484
Data\malignant\SOB_M_DC-14-10926-100-006.png
2485
Data\malignant\SOB_M_DC-14-10926-100-007.png
2486
Data\malignant\SOB_M_DC-14-10926-100-008.png
2487
Data\malignant\SOB_M_DC-14-10926-100-009.png
2488
Data\malignant\SOB_M_DC-14-10926-100-010.png
2489
Data\malignant\SOB_M_DC-14-10926-200-001.png
2490
Data\malignant\SOB_M_DC-14-10926-200-002.png
2491
Data\malignant\SOB_M_DC-14-10926-200-003.png
2492
Data\malignant\SOB_M_DC-14-10926-200-004.png
2493
Data\malignant\SOB_M_DC-14-10926-200-005.png
2494
Data\malignant\SOB_M_DC-14-10926-200-006.png
2495
Data\malignant\SOB_M_DC-14-10926-200-007.png
2496
Data\malignant\SOB_M_DC-14-10926-200-008.png
2497
Data\malignant\SOB_M_DC-14-10926-200-009.png
2498
Data\malignant\SOB_M_DC-14-10926-40-001.png
2499
D

In [6]:
# Convert to df
df = pd.DataFrame(records)

In [7]:
df

,path,filename,label,label_name
0,Data\benign\SOB_B_A-14-22549AB-100-001.png,SOB_B_A-14-22549AB-100-001.png,0,benign
1,Data\benign\SOB_B_A-14-22549AB-100-002.png,SOB_B_A-14-22549AB-100-002.png,0,benign
2,Data\benign\SOB_B_A-14-22549AB-100-003.png,SOB_B_A-14-22549AB-100-003.png,0,benign
3,Data\benign\SOB_B_A-14-22549AB-100-004.png,SOB_B_A-14-22549AB-100-004.png,0,benign
4,Data\benign\SOB_B_A-14-22549AB-100-005.png,SOB_B_A-14-22549AB-100-005.png,0,benign
...,...,...,...,...
7778,Data\malignant\SOB_M_PC-15-190EF-400-011.png,SOB_M_PC-15-190EF-400-011.png,0,malignant
7779,Data\malignant\SOB_M_PC-15-190EF-400-012.png,SOB_M_PC-15-190EF-400-012.png,0,malignant
7780,Data\malignant\SOB_M_PC-15-190EF-400-013.png,SOB_M_PC-15-190EF-400-013.png,0,malignant
7781,Data\malignant\SOB_M_PC-15-190EF-400-014.png,SOB_M_PC-15-190EF-400-014.png,0,malignant


# Extracting patient ID

In [8]:
def extract_patient_id(filename):
    """
    Extract patient ID from filename.
    Example: SOB_B_A-14-22549AB-40-011.png → 14-22549AB
    """

    # Remove the .png extension
    name = filename.replace(".png","")

    # Pattern: SOB_[B/M]_[TYPE]-[YEAR]-[SLIDE]-[MAG]-[SEQ]
    match = re.search(r"SOB_[BM]_[A-Z]+-(\d+-[A-Z0-9]+)-\d+-\d", name)

    if match:
        return match.group(1)
    else: 
        return None

In [9]:
# Apply the function to every filename
df['patient_id'] = df['filename'].apply(extract_patient_id)

In [10]:
df = df[['patient_id', 'filename', 'label', 'label_name', 'path']]

In [11]:
df.tail(5)

,patient_id,filename,label,label_name,path
7778,15-190EF,SOB_M_PC-15-190EF-400-011.png,0,malignant,Data\malignant\SOB_M_PC-15-190EF-400-011.png
7779,15-190EF,SOB_M_PC-15-190EF-400-012.png,0,malignant,Data\malignant\SOB_M_PC-15-190EF-400-012.png
7780,15-190EF,SOB_M_PC-15-190EF-400-013.png,0,malignant,Data\malignant\SOB_M_PC-15-190EF-400-013.png
7781,15-190EF,SOB_M_PC-15-190EF-400-014.png,0,malignant,Data\malignant\SOB_M_PC-15-190EF-400-014.png
7782,15-190EF,SOB_M_PC-15-190EF-400-015.png,0,malignant,Data\malignant\SOB_M_PC-15-190EF-400-015.png


In [12]:
print("Number of images with missing patient_id:", df["patient_id"].isna().sum())
print("\nExample of extracted patient IDs:")
print(df[["patient_id", "filename"]].head(10))

Number of images with missing patient_id: 0

Example of extracted patient IDs:
   patient_id                        filename
0  14-22549AB  SOB_B_A-14-22549AB-100-001.png
1  14-22549AB  SOB_B_A-14-22549AB-100-002.png
2  14-22549AB  SOB_B_A-14-22549AB-100-003.png
3  14-22549AB  SOB_B_A-14-22549AB-100-004.png
4  14-22549AB  SOB_B_A-14-22549AB-100-005.png
5  14-22549AB  SOB_B_A-14-22549AB-100-006.png
6  14-22549AB  SOB_B_A-14-22549AB-100-007.png
7  14-22549AB  SOB_B_A-14-22549AB-100-008.png
8  14-22549AB  SOB_B_A-14-22549AB-100-009.png
9  14-22549AB  SOB_B_A-14-22549AB-100-010.png


# QuickEDA

In [13]:
print("Total unique patients:", df["patient_id"].nunique())

print("\nNumber of patients per class:")
print(df.groupby("label_name")["patient_id"].nunique())

print("\nAverage number of images per patient:")
print(df.groupby("patient_id").size().mean())

print("\nImages per patient (summary):")
print(df.groupby("patient_id").size().describe())

Total unique patients: 81

Number of patients per class:
label_name
benign       24
malignant    57
Name: patient_id, dtype: int64

Average number of images per patient:
96.08641975308642

Images per patient (summary):
count     81.000000
mean      96.086420
std       38.256763
min       38.000000
25%       64.000000
50%       90.000000
75%      125.000000
max      235.000000
dtype: float64


# Patient-level df

In [14]:
df_patient=(
    df.groupby("patient_id")
      .agg(
          label=("label", "first"),
          label_name=("label_name", "first"),
          n_images=("filename", "count")
      )
      .reset_index()
)

print(df_patient.head(5))
print(df_patient.tail(5))
print(f"Total patient: {len(df_patient)}")
print(df_patient["label_name"].value_counts())

  patient_id  label label_name  n_images
0   14-10147      0  malignant        64
1   14-10926      0  malignant        39
2   14-11031      0  malignant        60
3   14-11520      0  malignant        98
4   14-11951      0  malignant       100
   patient_id  label label_name  n_images
76    14-8168      0  malignant        38
77    14-9133      0     benign       126
78    14-9146      0  malignant        90
79    14-9461      0  malignant       155
80   15-190EF      0  malignant        65
Total patient: 81
label_name
malignant    57
benign       24
Name: count, dtype: int64


# Patient-level Split

In [15]:
df_patient=(
    df.groupby("patient_id")
      .agg(
          label=("label", "first"),
          label_name=("label_name", "first"),
          n_image=("filename", "count")
      )
      .reset_index()
)

print(df_patient.head(5))
print(df_patient.tail(5))

  patient_id  label label_name  n_image
0   14-10147      0  malignant       64
1   14-10926      0  malignant       39
2   14-11031      0  malignant       60
3   14-11520      0  malignant       98
4   14-11951      0  malignant      100
   patient_id  label label_name  n_image
76    14-8168      0  malignant       38
77    14-9133      0     benign      126
78    14-9146      0  malignant       90
79    14-9461      0  malignant      155
80   15-190EF      0  malignant       65


# Patient-level Split

In [16]:
train, temp = train_test_split(
    df_patient,
    test_size= 0.25,
    stratify= df_patient["label"],
    random_state=42
)

val, test = train_test_split(
    temp,
    test_size=0.60,              # 60% of 25% = 15%
    stratify=temp["label"],
    random_state=42
)

print(f"Train patients: {len(train)}")
print(f"Val patients:   {len(val)}")
print(f"Test patients:  {len(test)}")

Train patients: 60
Val patients:   8
Test patients:  13


# Expand back to images

In [17]:
train=df[df['patient_id'].isin(train['patient_id'])].reset_index(drop=True)
val=df[df['patient_id'].isin(val['patient_id'])].reset_index(drop=True)
test=df[df['patient_id'].isin(test['patient_id'])].reset_index(drop=True)

print(f"Train patients: {len(train)}")
print(f"Val patients:   {len(val)}")
print(f"Test patients:  {len(test)}")

Train patients: 5842
Val patients:   864
Test patients:  1077


In [18]:
# Converting patient IDs into sets
train_ids = set(train['patient_id'])
val_ids = set(val['patient_id'])
test_ids = set(test['patient_id'])

# Check Overlap
print(f"Overlap Train Val: {len(train_ids & val_ids)}")
print(f"Overlap Test Val: {len(test_ids & val_ids)}")
print(f"Overlap Train Test: {len(train_ids & test_ids)}")

Overlap Train Val: 0
Overlap Test Val: 0
Overlap Train Test: 0


# Dataset Class

In [19]:
class BreakHisDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load image
        image = Image.open(row["path"]).convert("RGB")
        label = row["label"]
        
        # Apply transformation if any
        if self.transform:
            image = self.transform(image)
            
        return image, label


# Transform data

In [20]:
from torchvision import transforms

# Augmentation + Transformation Training Dataset
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=90),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# Transformation only for Validation & Test
val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# Create Dataset Object

In [21]:
train_images = BreakHisDataset(train, transform=train_transform)
val_images = BreakHisDataset(val, transform=val_test_transform)
test_images = BreakHisDataset(test, transform=val_test_transform)

In [22]:
print(len(train_images))
print(len(val_images))
print(len(test_images))

5842
864
1077


# DataLoaders

In [23]:
from torch.utils.data import DataLoader

In [24]:
BATCH_SIZE = 32

In [28]:
train_loader = DataLoader(
    train_images,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_images,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_images,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [29]:
# Test one batch
images, labels = next(iter(train_loader))
print("Batch image shape:", images.shape)   # should be [32, 3, 224, 224]
print("Batch labels shape:", labels.shape)
print("Labels:", labels[:10])

Batch image shape: torch.Size([32, 3, 224, 224])
Batch labels shape: torch.Size([32])
Labels: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
